In [1]:
import pandas as pd
import numpy as np
import os
import json
from pathlib import Path
from datetime import datetime,  timezone

In [2]:

# Настройки путей
data_path = "../data/channels_data/raw/"
output_path = "../data/channels_data/interim/"
stats_path = "../data/stats/"

# Создаем папки для результатов и статистики
Path(output_path).mkdir(parents=True, exist_ok=True)
Path(stats_path).mkdir(parents=True, exist_ok=True)

# Создаем папку для результатов, если её нет
Path(output_path).mkdir(parents=True, exist_ok=True)

# Получаем список CSV файлов
csv_files = [f for f in os.listdir(data_path) if f.endswith('.csv')]

current_date = datetime(2025, 12, 1, tzinfo=timezone.utc)
# Начальная дата для фильтрации (1 декабря 2024) - с учетом часового пояса UTC
start_date = datetime(2024, 12, 1, tzinfo=timezone.utc)

print("Обработка всех файлов завершена!")


Обработка всех файлов завершена!


In [3]:
# Словарь для хранения статистики
all_stats = {}

for csv_file in csv_files:
    print(f"\n{'='*60}")
    print(f"Обработка: {csv_file}")
    print('='*60)
    
    # Читаем CSV файл
    file_path = os.path.join(data_path, csv_file)
    df = pd.read_csv(file_path)
    
    # Статистика по файлу
    file_stats = {
        'filename': csv_file,
        'total_rows': len(df)
    }
    
    # Преобразуем столбец с датой
    date_column = None
    if 'post_datetime' in df.columns:
        date_column = 'post_datetime'
        df['post_datetime'] = pd.to_datetime(df['post_datetime'])
    elif 'post_date' in df.columns and 'post_time' in df.columns:
        # Объединяем дату и время
        df['post_datetime'] = pd.to_datetime(df['post_date'] + ' ' + df['post_time'])
        date_column = 'post_datetime'
    elif 'post_date' in df.columns:
        df['post_date'] = pd.to_datetime(df['post_date'])
        date_column = 'post_date'
    
    if not date_column:
        print(f"  ⚠️ Нет столбца с датой. Пропускаем файл.")
        continue
    
    # Фильтруем данные за последний год
    df_filtered = df[df[date_column] >= start_date].copy()
    file_stats['filtered_rows'] = len(df_filtered)
    file_stats['filtered_percentage'] = (len(df_filtered) / len(df) * 100) if len(df) > 0 else 0
    
    print(f"  📊 Всего записей: {len(df):,}")
    print(f"  📅 За последний год: {len(df_filtered):,} ({file_stats['filtered_percentage']:.1f}%)")
    
    if len(df_filtered) == 0:
        print(f"  ⚠️ Нет данных за последний год. Создаем пустой файл.")
        df_filtered.to_csv(os.path.join(output_path, csv_file), index=False)
        all_stats[csv_file] = file_stats
        continue
    
    # Находим столбцы с реакциями
    reaction_columns = [col for col in df_filtered.columns if col.startswith('reaction_')]
    file_stats['total_reactions'] = len(reaction_columns)
    
    if not reaction_columns:
        print(f"  ℹ️ Нет столбцов с реакциями")
        df_filtered.to_csv(os.path.join(output_path, csv_file), index=False)
        all_stats[csv_file] = file_stats
        continue
    
    # Анализ реакций
    reaction_stats = []
    reactions_to_keep = []
    
    for reaction_col in reaction_columns:
        # Заменяем NaN на 0
        reaction_series = df_filtered[reaction_col].fillna(0)
        
        # Сумма реакций
        total_reactions = reaction_series.sum()
        
        # Количество постов с этой реакцией
        posts_with_reaction = (reaction_series > 0).sum()
        
        # Процент постов с этой реакцией
        percentage_posts = (posts_with_reaction / len(df_filtered)) * 100
        
        # Статистика для этой реакции
        reaction_stat = {
            'reaction': reaction_col,
            'total_count': int(total_reactions),
            'posts_with_reaction': int(posts_with_reaction),
            'percentage_of_posts': round(percentage_posts, 2)
        }
        
        # Если реакция использовалась, находим первый пост с ней
        if total_reactions > 0:
            reactions_to_keep.append(reaction_col)
            reaction_stat['status'] = 'kept'
            
            # Находим первый пост с этой реакцией
            first_post_mask = reaction_series > 0
            if first_post_mask.any():
                first_post_idx = first_post_mask.idxmax()
                first_post = df_filtered.loc[first_post_idx]
                
                # Сохраняем информацию о посте
                reaction_stat['example_post'] = {
                    'post_id': str(first_post.get('post_id', 'N/A')),
                    'post_datetime': str(first_post.get('post_datetime', 'N/A')),
                    'post_date': str(first_post.get('post_date', 'N/A')),
                    'post_time': str(first_post.get('post_time', 'N/A')),
                    'reaction_count': int(reaction_series.loc[first_post_idx]),
                    'text_preview': str(first_post.get('text', 'N/A'))[:100] + '...' if pd.notna(first_post.get('text')) and len(str(first_post.get('text'))) > 100 else str(first_post.get('text', 'N/A'))
                }
                
                # Альтернативно: находим пост с максимальным количеством этой реакции
                max_reaction_idx = reaction_series.idxmax()
                if max_reaction_idx != first_post_idx:
                    max_post = df_filtered.loc[max_reaction_idx]
                    reaction_stat['max_post'] = {
                        'post_id': str(max_post.get('post_id', 'N/A')),
                        'post_datetime': str(max_post.get('post_datetime', 'N/A')),
                        'reaction_count': int(reaction_series.loc[max_reaction_idx])
                    }
        else:
            reaction_stat['status'] = 'removed'
        
        reaction_stats.append(reaction_stat)
    
    # Сортируем по количеству использований
    reaction_stats.sort(key=lambda x: x['total_count'], reverse=True)
    
    # Сохраняем статистику
    file_stats['kept_reactions'] = len(reactions_to_keep)
    file_stats['removed_reactions'] = len(reaction_columns) - len(reactions_to_keep)
    file_stats['reaction_details'] = reaction_stats
    
    # Создаем финальный DataFrame
    non_reaction_columns = [col for col in df_filtered.columns if not col.startswith('reaction_')]
    columns_to_save = non_reaction_columns + reactions_to_keep
    df_final = df_filtered[columns_to_save].copy()
    
    # Сохраняем обработанный файл
    output_file_path = os.path.join(output_path, csv_file)
    df_final.to_csv(output_file_path, index=False)
    
    # Выводим подробную информацию о кастомных реакциях
    print(f"\n  🎭 ИНФОРМАЦИЯ О КАСТОМНЫХ РЕАКЦИЯХ:")
    print(f"  {'='*50}")
    
    # Фильтруем только кастомные реакции (содержащие 'custom' или имеющие числовой ID)
    custom_reactions = [r for r in reaction_stats if 'custom' in r['reaction'].lower() or any(char.isdigit() for char in r['reaction'])]
    
    if custom_reactions:
        print(f"  Всего кастомных реакций: {len(custom_reactions)}")
        print(f"  {'─'*50}")
        
        for i, reaction_stat in enumerate(custom_reactions, 1):
            reaction_name = reaction_stat['reaction']
            status = reaction_stat['status']
            
            if status == 'kept' and 'example_post' in reaction_stat:
                example = reaction_stat['example_post']
                print(f"  {i:2}. {reaction_name}")
                print(f"      📍 Статус: Сохранена (использована {reaction_stat['total_count']:,} раз)")
                print(f"      📅 Дата примера: {example.get('post_date', 'N/A')} {example.get('post_time', '')}")
                print(f"      🕒 Полная дата: {example.get('post_datetime', 'N/A')}")
                print(f"      🆔 ID поста: {example.get('post_id', 'N/A')}")
                print(f"      🔢 Кол-во реакций в посте: {example.get('reaction_count', 'N/A')}")
                if 'text_preview' in example:
                    print(f"      📝 Текст (превью): {example['text_preview']}")
                print(f"      {'─'*40}")
            elif status == 'removed':
                print(f"  {i:2}. {reaction_name}")
                print(f"      ❌ Статус: Удалена (не использовалась)")
                print(f"      {'─'*40}")
    else:
        print(f"  ℹ️ Кастомные реакции не найдены")
    
    # Выводим краткую статистику
    print(f"\n  📊 КРАТКАЯ СТАТИСТИКА КАНАЛА:")
    print(f"    Реакций всего: {len(reaction_columns)}")
    print(f"    Сохранено: {len(reactions_to_keep)}")
    print(f"    Удалено: {len(reaction_columns) - len(reactions_to_keep)}")
    
    # Топ-5 реакций
    print(f"\n  🏆 ТОП-5 РЕАКЦИЙ ПО ИСПОЛЬЗОВАНИЮ:")
    for i, stat in enumerate(reaction_stats[:5], 1):
        if stat['total_count'] > 0:
            example_info = ""
            if 'example_post' in stat:
                example = stat['example_post']
                example_info = f" | Пример: {example.get('post_date', 'N/A')} (ID: {example.get('post_id', 'N/A')})"
            
            print(f"    {i}. {stat['reaction']}: {stat['total_count']:,} раз "
                  f"({stat['percentage_of_posts']:.1f}% постов){example_info}")
    
    all_stats[csv_file] = file_stats

# Сохраняем общую статистику
stats_file = os.path.join(stats_path, 'processing_stats.json')
with open(stats_file, 'w', encoding='utf-8') as f:
    json.dump(all_stats, f, ensure_ascii=False, indent=2, default=str)

print(f"\n{'='*60}")
print("✅ ОБРАБОТКА ЗАВЕРШЕНА!")
print('='*60)

# Сводная статистика
total_files = len(csv_files)
processed_files = len([s for s in all_stats.values() if 'filtered_rows' in s])
print(f"\n📈 СВОДНАЯ СТАТИСТИКА:")
print(f"  📁 Файлов обработано: {processed_files}/{total_files}")
print(f"  📊 Данные сохранены в: {output_path}")
print(f"  📈 Статистика сохранена в: {stats_file}")

# Дополнительный анализ кастомных реакций
print(f"\n🔍 АНАЛИЗ КАСТОМНЫХ РЕАКЦИЙ ПО ВСЕМ КАНАЛАМ:")
print('='*80)

# Собираем все кастомные реакции из всех каналов
all_custom_reactions = []

for channel_name, channel_stats in all_stats.items():
    if 'reaction_details' in channel_stats:
        for reaction_stat in channel_stats['reaction_details']:
            # Определяем кастомные реакции по наличию 'custom' или цифр в названии
            reaction_name = reaction_stat['reaction']
            if ('custom' in reaction_name.lower() or 
                any(char.isdigit() for char in reaction_name)):
                
                reaction_stat['channel'] = channel_name
                all_custom_reactions.append(reaction_stat)

if all_custom_reactions:
    print(f"📋 Всего найдено кастомных реакций: {len(all_custom_reactions)}")
    print(f"{'='*80}")
    
    # Группируем по каналам
    channels_with_custom = {}
    for reaction in all_custom_reactions:
        channel = reaction['channel']
        if channel not in channels_with_custom:
            channels_with_custom[channel] = []
        channels_with_custom[channel].append(reaction)
    
    print(f"📊 РАСПРЕДЕЛЕНИЕ ПО КАНАЛАМ:")
    for channel, reactions in channels_with_custom.items():
        kept = sum(1 for r in reactions if r['status'] == 'kept')
        removed = sum(1 for r in reactions if r['status'] == 'removed')
        print(f"  📁 {channel}: {len(reactions)} кастомных реакций ({kept} сохранено, {removed} удалено)")
    
    print(f"\n📋 ПОДРОБНАЯ ИНФОРМАЦИЯ О КАЖДОЙ КАСТОМНОЙ РЕАКЦИИ:")
    print(f"{'='*80}")
    
    for i, reaction in enumerate(all_custom_reactions, 1):
        if reaction['status'] == 'kept' and 'example_post' in reaction:
            example = reaction['example_post']
            print(f"{i:3}. 📍 Канал: {reaction['channel']}")
            print(f"     🎭 Реакция: {reaction['reaction']}")
            print(f"     📊 Использований: {reaction['total_count']:,}")
            print(f"     📅 Дата примера: {example.get('post_date', 'N/A')}")
            print(f"     🕒 Время: {example.get('post_time', 'N/A')}")
            print(f"     🆔 ID поста: {example.get('post_id', 'N/A')}")
            print(f"     {'─'*60}")
    
    # Создаем отдельный файл с информацией о кастомных реакциях
    custom_reactions_file = os.path.join(stats_path, 'custom_reactions_detailed.csv')
    
    # Подготавливаем данные для CSV
    custom_data = []
    for reaction in all_custom_reactions:
        if reaction['status'] == 'kept' and 'example_post' in reaction:
            example = reaction['example_post']
            row = {
                'channel': reaction['channel'],
                'reaction_name': reaction['reaction'],
                'total_uses': reaction['total_count'],
                'posts_with_reaction': reaction['posts_with_reaction'],
                'percentage_of_posts': reaction['percentage_of_posts'],
                'example_post_id': example.get('post_id', 'N/A'),
                'example_post_date': example.get('post_date', 'N/A'),
                'example_post_time': example.get('post_time', 'N/A'),
                'example_post_datetime': example.get('post_datetime', 'N/A'),
                'reaction_count_in_post': example.get('reaction_count', 'N/A'),
                'status': reaction['status']
            }
            custom_data.append(row)
    
    # Сохраняем в CSV
    if custom_data:
        custom_df = pd.DataFrame(custom_data)
        custom_df.to_csv(custom_reactions_file, index=False, encoding='utf-8')
        print(f"\n💾 Подробная информация о кастомных реакциях сохранена в: {custom_reactions_file}")
        
        # Выводим сводную таблицу
        print(f"\n📊 СВОДНАЯ ТАБЛИЦА КАСТОМНЫХ РЕАКЦИЙ:")
        print(f"{'='*100}")
        
        # Создаем красивый вывод
        summary_df = custom_df[['channel', 'reaction_name', 'total_uses', 
                               'example_post_date', 'example_post_id']].copy()
        summary_df.columns = ['Канал', 'Название реакции', 'Всего использований', 
                             'Дата примера', 'ID поста']
        
        # Форматируем вывод
        pd.set_option('display.max_colwidth', 50)
        pd.set_option('display.width', 120)
        
        print(summary_df.to_string(index=False))
else:
    print("ℹ️ Кастомные реакции не найдены ни в одном канале")


Обработка: AI_point_of_view.csv
  📊 Всего записей: 787
  📅 За последний год: 394 (50.1%)

  🎭 ИНФОРМАЦИЯ О КАСТОМНЫХ РЕАКЦИЯХ:
  ℹ️ Кастомные реакции не найдены

  📊 КРАТКАЯ СТАТИСТИКА КАНАЛА:
    Реакций всего: 44
    Сохранено: 33
    Удалено: 11

  🏆 ТОП-5 РЕАКЦИЙ ПО ИСПОЛЬЗОВАНИЮ:
    1. reaction_🔥: 1,355 раз (70.3% постов) | Пример: 2025-11-28 (ID: 1813)
    2. reaction_👍: 1,137 раз (83.0% постов) | Пример: 2025-11-28 (ID: 1813)
    3. reaction_❤: 851 раз (55.1% постов) | Пример: 2025-11-28 (ID: 1805)
    4. reaction_👏: 351 раз (29.2% постов) | Пример: 2025-11-27 (ID: 1797)
    5. reaction_🎉: 246 раз (9.9% постов) | Пример: 2025-11-26 (ID: 1785)

Обработка: DSCSproAI.csv
  📊 Всего записей: 731
  📅 За последний год: 370 (50.6%)

  🎭 ИНФОРМАЦИЯ О КАСТОМНЫХ РЕАКЦИЯХ:
  Всего кастомных реакций: 18
  ──────────────────────────────────────────────────
   1. reaction_custom_5368748388885999718
      📍 Статус: Сохранена (использована 287 раз)
      📅 Дата примера: 2025-04-04 10:56:12
   

In [17]:
import pandas as pd
import numpy as np
import os
import re
from pathlib import Path

# Карта преобразования кастомных реакций в эмодзи
CUSTOM_REACTIONS_MAP = {
    'by_channel': {
        'ict_moscow_ai': {
            '5217962652742996135': '❤️',
            '5289509472090220232': '❤️',
            '5445042860787252192': '❤️',
        },
        'd_code': {
            '5348351516182857544': '❤️',
            '5350663853560581569': '❤️',
        },
        'DSCSproAI': {
            '5436189026224728788': '🎉',
            '5438237047020088735': '🤖',
            '5235807206770750407': '😎',
            '5458801195814508191': '🤩',
            '5435931719028997627': '🩷',
            '5368748388885999718': '🩷',
            '4960832663062578456': '🩷',
            '5307602562990492983': '🔥',
            '5472411062412254753': '🧑‍🎓',
            '5435968389459767071': '🔥',
            '5436383132976702162': '⭐',
            '5438544137181752811': '⚡',
            '4961156112754672853': '⚡',
            '5323772371830588991': '🫡',
            '5361918153934791430': '🩷',
            '5424815192615706042': '🩷',
            '5818845400840277279': '👀',
            '4961121357879313223': '🩷',
            '6032706485526465658': '🩷',
            '5235826817591420001': '👨‍💻',
            '5395732581780040886': '🤝',
        },
        'seeallochnaya': {
            '5355083340548419513': '🥹',
            '5248951709367017473': '🚫',
        },
        'spbuniversity1724': {
            '5422748398518301870': '❤️',
            '5420436396148027032': '❤️',
            '5411468637577965032': '❤️',
            '5307926626862909975': '❤️',
            '5388584785536904914': '❤️',
            '5388802501724092155': '❤️',
        },
        'naebnet': {
            '5235478122081560535': '👍',
        }
    }
}

# Настройки путей
input_path = "../data/interim/"
output_path = "../data/processed/"
Path(output_path).mkdir(parents=True, exist_ok=True)

def get_channel_name_from_filename(filename):
    """
    Извлекает имя канала из имени файла.
    Предполагается, что файлы имеют формат: channelname.csv или channelname_other.csv
    """
    # Удаляем расширение .csv
    base_name = filename.replace('.csv', '')
    
    # Проверяем соответствие с ключами в карте
    for channel_name in CUSTOM_REACTIONS_MAP['by_channel'].keys():
        # Если имя файла начинается с имени канала
        if base_name.startswith(channel_name):
            return channel_name
    
    # Если не нашли точное соответствие, пробуем найти частичное
    for channel_name in CUSTOM_REACTIONS_MAP['by_channel'].keys():
        if channel_name in base_name:
            return channel_name
    
    return None

def process_custom_reactions(df, channel_name):
    """
    Преобразует кастомные реакции в эмодзи и суммирует их
    """
    if channel_name not in CUSTOM_REACTIONS_MAP['by_channel']:
        print(f"  ⚠️ Для канала '{channel_name}' нет карты преобразования")
        return df.copy()
    
    # Создаем копию DataFrame для изменений
    df_processed = df.copy()
    
    # Получаем карту преобразований для данного канала
    channel_map = CUSTOM_REACTIONS_MAP['by_channel'][channel_name]
    
    print(f"  📋 Карта преобразований для канала '{channel_name}':")
    for custom_id, emoji in channel_map.items():
        print(f"    {custom_id} → {emoji}")
    
    # Словарь для отслеживания преобразований
    transformations = []
    
    # Обрабатываем каждую кастомную реакцию из карты
    for custom_id, target_emoji in channel_map.items():
        # Ищем столбец с кастомной реакцией
        custom_column = None
        
        # Варианты названий столбца
        possible_columns = [
            f"reaction_custom_{custom_id}",
            f"reaction_{custom_id}",
            custom_id
        ]
        
        for col_name in possible_columns:
            if col_name in df_processed.columns:
                custom_column = col_name
                break
        
        if custom_column is None:
            print(f"  ⚠️ Столбец для кастомной реакции {custom_id} не найден")
            continue
        
        # Проверяем, есть ли уже столбец с целевым эмодзи
        emoji_column = f"reaction_{target_emoji}"
        
        # Если есть столбец с кастомной реакцией
        if custom_column in df_processed.columns:
            # Суммируем значения с целевым эмодзи
            if emoji_column in df_processed.columns:
                # Суммируем существующие значения
                df_processed[emoji_column] = df_processed[emoji_column].fillna(0) + df_processed[custom_column].fillna(0)
                transformation_type = "суммировано с существующим"
            else:
                # Создаем новый столбец с эмодзи
                df_processed[emoji_column] = df_processed[custom_column].fillna(0)
                transformation_type = "создан новый столбец"
            
            # Удаляем кастомный столбец
            df_processed = df_processed.drop(columns=[custom_column])
            
            # Собираем статистику
            total_reactions = df_processed[emoji_column].sum()
            posts_with_reaction = (df_processed[emoji_column] > 0).sum()
            
            transformations.append({
                'custom_id': custom_id,
                'custom_column': custom_column,
                'emoji': target_emoji,
                'emoji_column': emoji_column,
                'type': transformation_type,
                'total_reactions': int(total_reactions),
                'posts_with_reaction': int(posts_with_reaction)
            })
    
    # Выводим статистику преобразований
    if transformations:
        print(f"\n  🔄 ПРЕОБРАЗОВАНИЯ РЕАКЦИЙ:")
        print(f"  {'─'*60}")
        
        for i, trans in enumerate(transformations, 1):
            print(f"  {i:2}. {trans['custom_id']} → {trans['emoji']}")
            print(f"      Тип: {trans['type']}")
            print(f"      Всего реакций: {trans['total_reactions']:,}")
            print(f"      Постов с реакцией: {trans['posts_with_reaction']}")
            print(f"      Столбец: {trans['custom_column']} → {trans['emoji_column']}")
            print(f"      {'─'*40}")
    
    return df_processed

def process_all_files():
    """
    Обрабатывает все файлы в папке interim
    """
    # Получаем список CSV файлов
    csv_files = sorted([f for f in os.listdir(input_path) if f.endswith('.csv')])
    
    print(f"📁 ОБРАБОТКА ФАЙЛОВ ИЗ ПАПКИ: {input_path}")
    print(f"📊 Файлов для обработки: {len(csv_files)}")
    print("="*80)
    
    # Словарь для статистики
    stats = {}
    
    for csv_file in csv_files:
        print(f"\n📄 Обработка файла: {csv_file}")
        print(f"{'─'*60}")
        
        # Определяем имя канала из имени файла
        channel_name = get_channel_name_from_filename(csv_file)
        
        if not channel_name:
            print(f"  ⚠️ Не удалось определить канал для файла {csv_file}")
            print(f"  ℹ️ Проверьте соответствие имен файлов и каналов в карте")
            continue
        
        print(f"  🏷️ Определен канал: {channel_name}")
        
        # Читаем файл
        file_path = os.path.join(input_path, csv_file)
        df = pd.read_csv(file_path)
        
        # Получаем исходную статистику по реакциям
        reaction_cols_before = [col for col in df.columns if col.startswith('reaction_')]
        print(f"  📊 Столбцов с реакциями до обработки: {len(reaction_cols_before)}")
        
        # Обрабатываем кастомные реакции
        df_processed = process_custom_reactions(df, channel_name)
        
        # Получаем статистику после обработки
        reaction_cols_after = [col for col in df_processed.columns if col.startswith('reaction_')]
        print(f"  📊 Столбцов с реакциями после обработки: {len(reaction_cols_after)}")
        
        # Сохраняем статистику
        stats[csv_file] = {
            'channel': channel_name,
            'reactions_before': len(reaction_cols_before),
            'reactions_after': len(reaction_cols_after),
            'reactions_removed': len(reaction_cols_before) - len(reaction_cols_after),
            'custom_reactions_found': len([col for col in reaction_cols_before if 'custom' in col]),
            'custom_reactions_processed': len(CUSTOM_REACTIONS_MAP['by_channel'].get(channel_name, {}))
        }
        
        # Сохраняем обработанный файл
        output_file_path = os.path.join(output_path, csv_file)
        df_processed.to_csv(output_file_path, index=False)
        print(f"  💾 Файл сохранен: {output_file_path}")
        
        # Выводим сравнение столбцов
        print(f"\n  🔍 СРАВНЕНИЕ СТОЛБЦОВ С РЕАКЦИЯМИ:")
        print(f"  {'─'*60}")
        
        # Находим удаленные столбцы
        removed_cols = set(reaction_cols_before) - set(reaction_cols_after)
        added_cols = set(reaction_cols_after) - set(reaction_cols_before)
        
        if removed_cols:
            print(f"  ❌ Удалено столбцов: {len(removed_cols)}")
            for col in sorted(removed_cols)[:10]:  # Показываем первые 10
                print(f"    - {col}")
            if len(removed_cols) > 10:
                print(f"    ... и еще {len(removed_cols) - 10} столбцов")
        
        if added_cols:
            print(f"\n  ✅ Добавлено столбцов: {len(added_cols)}")
            for col in sorted(added_cols):
                print(f"    + {col}")
        
        # Показываем топ-5 эмодзи по количеству реакций
        emoji_columns = [col for col in reaction_cols_after if any(emoji in col for emoji in ['❤️', '🔥', '👍', '🎉', '🤖'])]
        if emoji_columns:
            print(f"\n  🏆 ТОП-5 ЭМОДЗИ ПО КОЛИЧЕСТВУ РЕАКЦИЙ:")
            print(f"  {'─'*60}")
            
            emoji_stats = []
            for col in emoji_columns:
                total = df_processed[col].sum()
                emoji_stats.append((col, total))
            
            # Сортируем по убыванию
            emoji_stats.sort(key=lambda x: x[1], reverse=True)
            
            for i, (col, total) in enumerate(emoji_stats[:5], 1):
                emoji_name = col.replace('reaction_', '')
                print(f"  {i:2}. {emoji_name}: {int(total):,} реакций")
    
    return stats

# Запускаем обработку
print("🚀 ЗАПУСК ПРЕОБРАЗОВАНИЯ КАСТОМНЫХ РЕАКЦИЙ")
print("="*80)

stats = process_all_files()

# Выводим итоговую статистику
print(f"\n{'='*80}")
print("✅ ПРЕОБРАЗОВАНИЕ ЗАВЕРШЕНО!")
print(f"{'='*80}")

if stats:
    print(f"\n📈 ИТОГОВАЯ СТАТИСТИКА:")
    print(f"{'─'*80}")
    
    total_files = len(stats)
    total_reactions_before = sum(s['reactions_before'] for s in stats.values())
    total_reactions_after = sum(s['reactions_after'] for s in stats.values())
    total_custom_found = sum(s['custom_reactions_found'] for s in stats.values())
    total_custom_processed = sum(s['custom_reactions_processed'] for s in stats.values())
    
    print(f"📁 Обработано файлов: {total_files}")
    print(f"🎭 Всего столбцов с реакциями:")
    print(f"   До обработки: {total_reactions_before}")
    print(f"   После обработки: {total_reactions_after}")
    print(f"   Удалено: {total_reactions_before - total_reactions_after}")
    print(f"\n🔧 Кастомные реакции:")
    print(f"   Найдено в файлах: {total_custom_found}")
    print(f"   Обработано по карте: {total_custom_processed}")
    
    print(f"\n📊 СТАТИСТИКА ПО КАНАЛАМ:")
    print(f"{'─'*80}")
    
    headers = ["Канал", "Файл", "Реакций до", "Реакций после", "Кастомных", "Обработано"]
    print(f"{headers[0]:<20} {headers[1]:<30} {headers[2]:<12} {headers[3]:<12} {headers[4]:<10} {headers[5]:<10}")
    print(f"{'─'*80}")
    
    for filename, stat in stats.items():
        channel = stat['channel']
        reactions_before = stat['reactions_before']
        reactions_after = stat['reactions_after']
        custom_found = stat['custom_reactions_found']
        custom_processed = stat['custom_reactions_processed']
        
        print(f"{channel:<20} {filename:<30} {reactions_before:<12} {reactions_after:<12} {custom_found:<10} {custom_processed:<10}")

🚀 ЗАПУСК ПРЕОБРАЗОВАНИЯ КАСТОМНЫХ РЕАКЦИЙ
📁 ОБРАБОТКА ФАЙЛОВ ИЗ ПАПКИ: ../data/interim/
📊 Файлов для обработки: 10

📄 Обработка файла: AI_point_of_view.csv
────────────────────────────────────────────────────────────
  ⚠️ Не удалось определить канал для файла AI_point_of_view.csv
  ℹ️ Проверьте соответствие имен файлов и каналов в карте

📄 Обработка файла: DSCSproAI.csv
────────────────────────────────────────────────────────────
  🏷️ Определен канал: DSCSproAI
  📊 Столбцов с реакциями до обработки: 53
  📋 Карта преобразований для канала 'DSCSproAI':
    5436189026224728788 → 🎉
    5438237047020088735 → 🤖
    5235807206770750407 → 😎
    5458801195814508191 → 🤩
    5435931719028997627 → 🩷
    5368748388885999718 → 🩷
    4960832663062578456 → 🩷
    5307602562990492983 → 🔥
    5472411062412254753 → 🧑‍🎓
    5435968389459767071 → 🔥
    5436383132976702162 → ⭐
    5438544137181752811 → ⚡
    4961156112754672853 → ⚡
    5323772371830588991 → 🫡
    5361918153934791430 → 🩷
    54248151926157060

In [21]:
import pandas as pd
import numpy as np
import os
import re
from pathlib import Path

# Карта преобразования кастомных реакций в эмодзи
CUSTOM_REACTIONS_MAP = {
    'by_channel': {
        'ict_moscow_ai': {
            '5217962652742996135': '❤️',
            '5289509472090220232': '❤️',
            '5445042860787252192': '❤️',
        },
        'd_code': {
            '5348351516182857544': '❤️',
            '5350663853560581569': '❤️',
        },
        'DSCSproAI': {
            '5436189026224728788': '🎉',
            '5438237047020088735': '🤖',
            '5235807206770750407': '😎',
            '5458801195814508191': '🤩',
            '5435931719028997627': '❤️',
            '5368748388885999718': '❤️',
            '4960832663062578456': '❤️',
            '5307602562990492983': '🔥',
            '5472411062412254753': '🧑‍🎓',
            '5435968389459767071': '🔥',
            '5436383132976702162': '⭐',
            '5438544137181752811': '⚡',
            '4961156112754672853': '⚡',
            '5323772371830588991': '🫡',
            '5361918153934791430': '❤️',
            '5424815192615706042': '❤️',
            '5818845400840277279': '👀',
            '4961121357879313223': '❤️',
            '6032706485526465658': '❤️',
            '5235826817591420001': '👨‍💻',
            '5395732581780040886': '🤝',
        },
        'seeallochnaya': {
            '5355083340548419513': '🥹',
            '5248951709367017473': '🚫',
        },
        'spbuniversity1724': {
            '5422748398518301870': '❤️',
            '5420436396148027032': '❤️',
            '5411468637577965032': '❤️',
            '5307926626862909975': '❤️',
            '5388584785536904914': '❤️',
            '5388802501724092155': '❤️',
        },
        'naebnet': {
            '5235478122081560535': '👍',
        }
    }
}

# Список всех вариаций сердечек для стандартизации
HEART_VARIATIONS = [
    '❤️',  # красное сердце (основное)
    '🩷',  # розовое сердце
    '💘',  # сердце со стрелой
    '❤‍🔥',  # горящее сердце
    '❤',   # сердце без варианта символа
]

# Список всех каналов для обработки
ALL_CHANNELS = [
    'Education_Yandex',
    'skoltech_daily',
    'spbuniversity1724',
    'AI_point_of_view',
    'seeallochnaya',
    'tbank_education',
    'd_code',
    'naebnet',
    'ict_moscow_ai',
    'DSCSproAI'
]

# Настройки путей
input_path = "../data/processed/"
output_path = "../data/processed2/"
Path(output_path).mkdir(parents=True, exist_ok=True)

def get_channel_name_from_filename(filename):
    """
    Извлекает имя канала из имени файла.
    Ищет точное или частичное соответствие с известными каналами
    """
    # Удаляем расширение .csv
    base_name = filename.replace('.csv', '').lower()
    
    # Проверяем все возможные каналы
    for channel_name in ALL_CHANNELS:
        if channel_name.lower() in base_name:
            return channel_name
    
    # Если не нашли, пробуем найти по частям имени
    for channel_name in ALL_CHANNELS:
        # Убираем подчеркивания для более гибкого сравнения
        channel_simple = channel_name.replace('_', '').lower()
        base_simple = base_name.replace('_', '')
        if channel_simple in base_simple or base_simple in channel_simple:
            return channel_name
    
    # Если все еще не нашли, возвращаем имя файла без расширения
    return filename.replace('.csv', '')

def standardize_hearts(df, channel_name):
    """
    Стандартизирует все вариации сердечек в один вид ❤️ и суммирует их
    Эта функция вызывается для ВСЕХ каналов
    """
    df_processed = df.copy()
    
    # Находим все столбцы с сердечками
    heart_columns = []
    for heart_var in HEART_VARIATIONS:
        # Ищем столбцы с этой вариацией сердечка
        heart_cols = [col for col in df_processed.columns 
                     if col.startswith(f'reaction_{heart_var}')]
        heart_columns.extend(heart_cols)
    
    if not heart_columns:
        print(f"  ℹ️ Вариации сердечек не найдены")
        return df_processed, []
    
    print(f"  💖 Найдено вариаций сердечек: {len(heart_columns)}")
    
    # Создаем столбец для стандартизированного сердечка, если его еще нет
    standard_heart_col = 'reaction_❤️'
    
    # Если основного столбца еще нет, создаем его с нулями
    if standard_heart_col not in df_processed.columns:
        df_processed[standard_heart_col] = 0
    
    transformations = []
    heart_stats = []
    
    # Суммируем все сердечки в основной столбец
    for heart_col in heart_columns:
        if heart_col != standard_heart_col:  # Не суммируем основной столбец с самим собой
            # Получаем эмодзи из названия столбца
            emoji_match = re.search(r'reaction_(.+)$', heart_col)
            original_emoji = emoji_match.group(1) if emoji_match else heart_col
            
            # Суммируем значения
            heart_values = df_processed[heart_col].fillna(0)
            total_reactions = heart_values.sum()
            posts_with_heart = (heart_values > 0).sum()
            
            if total_reactions > 0:
                # Добавляем к основному столбцу
                df_processed[standard_heart_col] = df_processed[standard_heart_col].fillna(0) + heart_values
                
                # Записываем статистику
                heart_stats.append({
                    'original_column': heart_col,
                    'original_emoji': original_emoji,
                    'total_reactions': int(total_reactions),
                    'posts_with_reaction': int(posts_with_heart)
                })
                
                print(f"    {original_emoji} → ❤️: {int(total_reactions):,} реакций")
            else:
                heart_stats.append({
                    'original_column': heart_col,
                    'original_emoji': original_emoji,
                    'total_reactions': 0,
                    'posts_with_reaction': 0
                })
    
    # Удаляем все нестандартные столбцы с сердечками
    columns_to_remove = [col for col in heart_columns if col != standard_heart_col]
    if columns_to_remove:
        df_processed = df_processed.drop(columns=columns_to_remove)
        print(f"  ❌ Удалено столбцов с сердечками: {len(columns_to_remove)}")
    
    # Добавляем статистику по итоговому сердечку
    if standard_heart_col in df_processed.columns:
        final_total = df_processed[standard_heart_col].sum()
        final_posts = (df_processed[standard_heart_col] > 0).sum()
        transformations.append({
            'type': 'heart_standardization',
            'final_column': standard_heart_col,
            'final_total': int(final_total),
            'final_posts': int(final_posts),
            'merged_variations': len(heart_stats),
            'details': heart_stats
        })
    
    return df_processed, transformations

def process_custom_reactions(df, channel_name):
    """
    Преобразует кастомные реакции в эмодзи и суммирует их
    Вызывается только для каналов, которые есть в карте
    """
    if channel_name not in CUSTOM_REACTIONS_MAP['by_channel']:
        # Если канала нет в карте, просто возвращаем DataFrame без изменений
        return df.copy(), []
    
    # Создаем копию DataFrame для изменений
    df_processed = df.copy()
    
    # Получаем карту преобразований для данного канала
    channel_map = CUSTOM_REACTIONS_MAP['by_channel'][channel_name]
    
    print(f"  📋 Карта преобразований для канала '{channel_name}':")
    for custom_id, emoji in channel_map.items():
        print(f"    {custom_id} → {emoji}")
    
    # Словарь для отслеживания преобразований
    transformations = []
    
    # Обрабатываем каждую кастомную реакцию из карты
    for custom_id, target_emoji in channel_map.items():
        # Ищем столбец с кастомной реакцией
        custom_column = None
        
        # Варианты названий столбца
        possible_columns = [
            f"reaction_custom_{custom_id}",
            f"reaction_{custom_id}",
            custom_id
        ]
        
        for col_name in possible_columns:
            if col_name in df_processed.columns:
                custom_column = col_name
                break
        
        if custom_column is None:
            print(f"  ⚠️ Столбец для кастомной реакции {custom_id} не найден")
            continue
        
        # Проверяем, есть ли уже столбец с целевым эмодзи
        emoji_column = f"reaction_{target_emoji}"
        
        # Если есть столбец с кастомной реакцией
        if custom_column in df_processed.columns:
            # Суммируем значения с целевым эмодзи
            if emoji_column in df_processed.columns:
                # Суммируем существующие значения
                df_processed[emoji_column] = df_processed[emoji_column].fillna(0) + df_processed[custom_column].fillna(0)
                transformation_type = "суммировано с существующим"
            else:
                # Создаем новый столбец с эмодзи
                df_processed[emoji_column] = df_processed[custom_column].fillna(0)
                transformation_type = "создан новый столбец"
            
            # Удаляем кастомный столбец
            df_processed = df_processed.drop(columns=[custom_column])
            
            # Собираем статистику
            total_reactions = df_processed[emoji_column].sum()
            posts_with_reaction = (df_processed[emoji_column] > 0).sum()
            
            transformations.append({
                'type': 'custom_to_emoji',
                'custom_id': custom_id,
                'custom_column': custom_column,
                'emoji': target_emoji,
                'emoji_column': emoji_column,
                'transformation_type': transformation_type,
                'total_reactions': int(total_reactions),
                'posts_with_reaction': int(posts_with_reaction)
            })
    
    return df_processed, transformations

def process_file(csv_file):
    """
    Обрабатывает один файл: стандартизирует сердечки и преобразует кастомные реакции (если есть)
    """
    print(f"\n📄 Обработка файла: {csv_file}")
    print(f"{'─'*60}")
    
    # Определяем имя канала из имени файла
    channel_name = get_channel_name_from_filename(csv_file)
    
    if not channel_name:
        print(f"  ⚠️ Не удалось определить канал для файла {csv_file}")
        print(f"  ℹ️ Использую имя файла как идентификатор")
        channel_name = csv_file.replace('.csv', '')
    
    print(f"  🏷️ Канал: {channel_name}")
    
    # Читаем файл
    file_path = os.path.join(input_path, csv_file)
    df = pd.read_csv(file_path)
    
    # Получаем исходную статистику по реакциям
    reaction_cols_before = [col for col in df.columns if col.startswith('reaction_')]
    print(f"  📊 Столбцов с реакциями до обработки: {len(reaction_cols_before)}")
    
    all_transformations = []
    
    # Шаг 1: Стандартизируем сердечки (для ВСЕХ каналов)
    df_step1, heart_transformations = standardize_hearts(df, channel_name)
    all_transformations.extend(heart_transformations)
    
    # Шаг 2: Преобразуем кастомные реакции (только если канал есть в карте)
    df_final, custom_transformations = process_custom_reactions(df_step1, channel_name)
    all_transformations.extend(custom_transformations)
    
    # Получаем статистику после обработки
    reaction_cols_after = [col for col in df_final.columns if col.startswith('reaction_')]
    print(f"  📊 Столбцов с реакций после обработки: {len(reaction_cols_after)}")
    
    # Собираем статистику по файлу
    stats = {
        'channel': channel_name,
        'reactions_before': len(reaction_cols_before),
        'reactions_after': len(reaction_cols_after),
        'reactions_removed': len(reaction_cols_before) - len(reaction_cols_after),
        'transformations': all_transformations,
        'heart_variations_before': len([col for col in reaction_cols_before 
                                       if any(h in col for h in HEART_VARIATIONS)]),
        'heart_variations_after': len([col for col in reaction_cols_after 
                                      if any(h in col for h in HEART_VARIATIONS)]),
        'has_custom_map': channel_name in CUSTOM_REACTIONS_MAP['by_channel']
    }
    
    # Сохраняем обработанный файл
    output_file_path = os.path.join(output_path, csv_file)
    df_final.to_csv(output_file_path, index=False)
    print(f"  💾 Файл сохранен: {output_file_path}")
    
    return df_final, stats

def process_all_files():
    """
    Обрабатывает все файлы в папке interim
    """
    # Получаем список CSV файлов
    csv_files = sorted([f for f in os.listdir(input_path) if f.endswith('.csv')])
    
    print(f"📁 ОБРАБОТКА ФАЙЛОВ ИЗ ПАПКИ: {input_path}")
    print(f"📊 Файлов для обработки: {len(csv_files)}")
    print("="*80)
    
    # Словарь для статистики
    all_stats = {}
    
    for csv_file in csv_files:
        df_processed, file_stats = process_file(csv_file)
        
        if df_processed is not None:
            all_stats[csv_file] = file_stats
            
            # Выводим подробную информацию о преобразованиях
            if file_stats.get('transformations'):
                print(f"\n  🔄 ДЕТАЛИ ПРЕОБРАЗОВАНИЙ:")
                print(f"  {'─'*60}")
                
                for transform in file_stats.get('transformations', []):
                    if transform['type'] == 'custom_to_emoji':
                        print(f"  • Кастомная: {transform['custom_id']} → {transform['emoji']}")
                        print(f"    Всего реакций: {transform['total_reactions']:,}")
                    elif transform['type'] == 'heart_standardization':
                        print(f"  • Стандартизация сердечек:")
                        print(f"    Итоговый столбец: {transform['final_column']}")
                        print(f"    Объединено вариаций: {transform['merged_variations']}")
                        print(f"    Итоговое количество: {transform['final_total']:,}")
                        if transform.get('details'):
                            for detail in transform['details']:
                                if detail['total_reactions'] > 0:
                                    print(f"      - {detail['original_emoji']}: {detail['total_reactions']:,}")
            
            # Выводим итоговые результаты
            print(f"\n  🔍 ИТОГОВЫЕ РЕЗУЛЬТАТЫ:")
            print(f"  {'─'*60}")
            
            # Находим все столбцы с реакциями в обработанном файле
            reaction_cols = [col for col in df_processed.columns if col.startswith('reaction_')]
            
            # Группируем реакции по категориям
            heart_cols = [col for col in reaction_cols if '❤' in col]
            other_cols = [col for col in reaction_cols if '❤' not in col]
            
            if heart_cols:
                print(f"  💖 Сердечки: {len(heart_cols)} столбцов")
                for col in heart_cols:
                    total = df_processed[col].sum()
                    emoji = col.replace('reaction_', '')
                    posts = (df_processed[col] > 0).sum()
                    print(f"    {emoji}: {int(total):,} реакций в {posts} постах")
            else:
                print(f"  💖 Сердечки: не найдены")
            
            print(f"\n  🎭 Прочие реакции: {len(other_cols)} столбцов")
            
            # Топ-5 не-сердечек
            other_stats = []
            for col in other_cols:
                total = df_processed[col].sum()
                if total > 0:
                    emoji = col.replace('reaction_', '')
                    other_stats.append((emoji, total))
            
            if other_stats:
                other_stats.sort(key=lambda x: x[1], reverse=True)
                
                for i, (emoji, total) in enumerate(other_stats[:5], 1):
                    print(f"    {i:2}. {emoji}: {int(total):,} реакций")
            else:
                print(f"    ℹ️ Нет реакций с положительными значениями")
    
    return all_stats

# Запускаем обработку
print("🚀 ЗАПУСК ОБРАБОТКИ ВСЕХ ФАЙЛОВ")
print("="*80)

all_stats = process_all_files()

# Выводим итоговую статистику
print(f"\n{'='*80}")
print("✅ ОБРАБОТКА ВСЕХ ФАЙЛОВ ЗАВЕРШЕНА!")
print(f"{'='*80}")

if all_stats:
    print(f"\n📈 ИТОГОВАЯ СТАТИСТИКА ПО ВСЕМ КАНАЛАМ:")
    print(f"{'─'*80}")
    
    total_files = len(all_stats)
    total_reactions_before = sum(s['reactions_before'] for s in all_stats.values())
    total_reactions_after = sum(s['reactions_after'] for s in all_stats.values())
    total_heart_variations_before = sum(s['heart_variations_before'] for s in all_stats.values())
    total_heart_variations_after = sum(s['heart_variations_after'] for s in all_stats.values())
    files_with_custom_map = sum(1 for s in all_stats.values() if s['has_custom_map'])
    
    print(f"📁 Всего файлов: {total_files}")
    print(f"🎭 Столбцов с реакциями:")
    print(f"   До обработки: {total_reactions_before}")
    print(f"   После обработки: {total_reactions_after}")
    print(f"   Удалено: {total_reactions_before - total_reactions_after}")
    print(f"\n💖 Вариаций сердечек:")
    print(f"   До обработки: {total_heart_variations_before}")
    print(f"   После обработки: {total_heart_variations_after}")
    print(f"   Объединено: {total_heart_variations_before - total_heart_variations_after}")
    print(f"\n🗺️ Файлов с картой кастомных реакций: {files_with_custom_map}/{total_files}")
    
    print(f"\n📊 ДЕТАЛЬНАЯ СТАТИСТИКА ПО КАНАЛАМ:")
    print(f"{'─'*80}")
    
    headers = ["Канал", "Файл", "Реакций до", "Реакций после", "Сердечек до", "Сердечек после", "Карта"]
    print(f"{headers[0]:<20} {headers[1]:<30} {headers[2]:<12} {headers[3]:<12} {headers[4]:<12} {headers[5]:<12} {headers[6]:<8}")
    print(f"{'─'*80}")
    
    for filename, stat in all_stats.items():
        channel = stat['channel']
        reactions_before = stat['reactions_before']
        reactions_after = stat['reactions_after']
        hearts_before = stat['heart_variations_before']
        hearts_after = stat['heart_variations_after']
        has_map = "✓" if stat['has_custom_map'] else "✗"
        
        print(f"{channel:<20} {filename:<30} {reactions_before:<12} {reactions_after:<12} {hearts_before:<12} {hearts_after:<12} {has_map:<8}")

🚀 ЗАПУСК ОБРАБОТКИ ВСЕХ ФАЙЛОВ
📁 ОБРАБОТКА ФАЙЛОВ ИЗ ПАПКИ: ../data/processed/
📊 Файлов для обработки: 10

📄 Обработка файла: AI_point_of_view.csv
────────────────────────────────────────────────────────────
  🏷️ Канал: AI_point_of_view
  📊 Столбцов с реакциями до обработки: 33
  💖 Найдено вариаций сердечек: 4
    💘 → ❤️: 10 реакций
    ❤‍🔥 → ❤️: 117 реакций
    ❤ → ❤️: 851 реакций
    ❤‍🔥 → ❤️: 117 реакций
  ❌ Удалено столбцов с сердечками: 4
  📊 Столбцов с реакций после обработки: 31
  💾 Файл сохранен: ../data/processed2/AI_point_of_view.csv

  🔄 ДЕТАЛИ ПРЕОБРАЗОВАНИЙ:
  ────────────────────────────────────────────────────────────
  • Стандартизация сердечек:
    Итоговый столбец: reaction_❤️
    Объединено вариаций: 4
    Итоговое количество: 1,095
      - 💘: 10
      - ❤‍🔥: 117
      - ❤: 851
      - ❤‍🔥: 117

  🔍 ИТОГОВЫЕ РЕЗУЛЬТАТЫ:
  ────────────────────────────────────────────────────────────
  💖 Сердечки: 1 столбцов
    ❤️: 1,095 реакций в 231 постах

  🎭 Прочие реакции: 30 ст

In [4]:
import pandas as pd
# Создаем DataFrame со структурой
data = {
    'Группа признаков': [
        'Метаданные канала', 'Метаданные канала', 'Метаданные канала',
        'Метаданные публикации', 'Метаданные публикации', 'Метаданные публикации', 'Метаданные публикации',
        'Характеристики контента', 'Характеристики контента', 'Характеристики контента',
        'Характеристики контента', 'Характеристики контента', 'Характеристики контента', 'Характеристики контента',
        'Базовые метрики вовлечённости', 'Базовые метрики вовлечённости', 'Базовые метрики вовлечённости',
        'Детализированные реакции', 'Детализированные реакции', 'Детализированные реакции',
        'Детализированные реакции', 'Детализированные реакции', 'Детализированные реакции',
        'Детализированные реакции', 'Детализированные реакции', 'Детализированные реакции',
        'Детализированные реакции'
    ],
    'Признак': [
        'channel_username', 'channel_title', 'channel_subscribers',
        'post_id', 'post_date', 'post_time', 'post_datetime',
        'text', 'text_length', 'has_media', 'media_type', 'has_poll',
        'has_links_entities', 'links_count_entities',
        'views', 'forwards', 'total_reactions',
        'reaction_👍', 'reaction_❤️', 'reaction_🔥', 'reaction_😍', 
        'reaction_🎉', 'reaction_👏', 'reaction_🤔', 'reaction_😐',
        'reaction_👎', '... (всего 63 типа)'
    ],
    'Описание': [
        'Никнейм канала (уникальный ID в Telegram)',
        'Официальное название канала',
        'Количество подписчиков на момент сбора',
        'Уникальный идентификатор поста',
        'Дата публикации (YYYY-MM-DD)',
        'Время публикации (HH:MM:SS)',
        'Полная метка времени публикации',
        'Текст публикации',
        'Длина текста в символах',
        'Наличие медиавложения (True/False)',
        'Тип медиа: photo/video/document',
        'Наличие встроенного опроса',
        'Наличие гиперссылок в тексте',
        'Количество гиперссылок',
        'Количество просмотров',
        'Количество репостов (пересылок)',
        'Общее количество всех реакций',
        'Лайк (thumbs up)',
        'Сердце (red heart)',
        'Огонь (fire)',
        'Влюблённые глаза (heart eyes)',
        'Праздник (party popper)',
        'Аплодисменты (clapping hands)',
        'Размышление (thinking face)',
        'Нейтральное лицо (neutral face)',
        'Дизлайк (thumbs down)',
        'Всего 63 типа эмоциональных реакций в формате reaction_<emoji>'
    ]
}

df_structure = pd.DataFrame(data)

In [5]:
# Устанавливаем: pip install tabulate
from tabulate import tabulate

# Форматируем как красивую текстовую таблицу
table_data = []
for i, row in df_structure.iterrows():
    table_data.append([
        row['Группа признаков'],
        row['Признак'],
        row['Описание']
    ])

# Создаем красивый вывод
output = tabulate(table_data, 
                  headers=['Группа признаков', 'Признак', 'Описание'],
                  tablefmt='grid',
                  maxcolwidths=[25, 20, 50])

# Сохраняем в файл
with open('../docs/dataset_structure.txt', 'w', encoding='utf-8') as f:
    f.write("СТРУКТУРА ДАТАСЕТА ПУБЛИКАЦИЙ ИЗ TELEGRAM-КАНАЛОВ\n")
    f.write("=" * 100 + "\n\n")
    f.write(output)
    f.write(f"\n\nВсего признаков: 83 (включая 63 типа эмоциональных реакций)")

print("Текстовая таблица сохранена в '../docs/dataset_structure.txt'")
print(output[:500])  # Показываем первые 500 символов для предпросмотра

Текстовая таблица сохранена в 'dataset_structure_text.txt'
+--------------------------+----------------------+-----------------------------------------------+
| Группа признаков         | Признак              | Описание                                      |
+==========================+======================+===============================================+
| Метаданные канала        | channel_username     | Никнейм канала (уникальный ID в Telegram)     |
+--------------------------+----------------------+-----------------------------------------------+



In [ ]:
import plotly.graph_objects as go

# Создаем таблицу с Plotly
fig = go.Figure(data=[go.Table(
    columnwidth=[150, 150, 400],
    header=dict(
        values=['<b>Группа признаков</b>', '<b>Признак</b>', '<b>Описание</b>'],
        fill_color='#4CAF50',
        align='left',
        font=dict(color='white', size=12, family="Arial"),
        height=40
    ),
    cells=dict(
        values=[df_structure['Группа признаков'], 
                df_structure['Признак'], 
                df_structure['Описание']],
        fill_color=[
            ['#F1F8E9' if 'метаданные' in g.lower() else 
             '#FFF3E0' if 'контент' in g.lower() else
             '#FCE4EC' if 'метрики' in g.lower() else
             '#E3F2FD' if 'реакции' in g.lower() else
             'white' for g in df_structure['Группа признаков']]
        ],
        align='left',
        font=dict(size=11, family="Arial"),
        height=30
    )
)])

fig.update_layout(
    title=dict(
        text='<b>Структура датасета публикаций Telegram-каналов</b>',
        x=0.5,
        font=dict(size=16, family="Arial")
    ),
    width=1200,
    height=600,
    margin=dict(l=20, r=20, t=60, b=20)
)

# Сохраняем как HTML (интерактивный)
fig.write_html("dataset_structure_interactive.html")

# Или как изображение
fig.write_image("dataset_structure_plotly.png", width=1200, height=600)

print("Интерактивная таблица сохранена в 'dataset_structure_interactive.html'")
print("Изображение сохранено в 'dataset_structure_plotly.png'")